In [1]:
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("--- SENTRi-X: Transfer Learning (Domain Adaptation) ---")

processed_dir = "../data/processed/"
models_dir = "../models/"

# 1. Load the locally scaled BoT-IoT data
X_bot = joblib.load(os.path.join(processed_dir, "bot_iot_X_test.pkl"))
y_bot = joblib.load(os.path.join(processed_dir, "bot_iot_y_test.pkl"))

# 2. Split: 20% for "Studying" (Fine-Tuning), 80% for the Final Exam
X_study, X_exam, y_study, y_exam = train_test_split(X_bot, y_bot, test_size=0.80, random_state=42, stratify=y_bot)

print(f"Data reserved for SENTRi-X to study: {X_study.shape[0]:,} packets")
print(f"Data reserved for the Final Exam: {X_exam.shape[0]:,} packets")

--- SENTRi-X: Transfer Learning (Domain Adaptation) ---
Data reserved for SENTRi-X to study: 28,945 packets
Data reserved for the Final Exam: 115,783 packets


In [2]:
print("Initiating Transfer Learning on Random Forest...")

# Train a new RF using the 20% study data
rf_adapted = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_adapted.fit(X_study, y_study)

# Test it on the 80% unseen data
adapted_preds = rf_adapted.predict(X_exam)

print(f"\n--- SENTRi-X Performance Post-Transfer Learning ---")
print(f"Adapted Accuracy: {accuracy_score(y_exam, adapted_preds):.4%}")
print("\nDetailed Report:")
print(classification_report(y_exam, adapted_preds, target_names=['Normal', 'Attack']))

Initiating Transfer Learning on Random Forest...

--- SENTRi-X Performance Post-Transfer Learning ---
Adapted Accuracy: 99.9870%

Detailed Report:
              precision    recall  f1-score   support

      Normal       0.88      0.33      0.48        21
      Attack       1.00      1.00      1.00    115762

    accuracy                           1.00    115783
   macro avg       0.94      0.67      0.74    115783
weighted avg       1.00      1.00      1.00    115783

